# Summary
This notebook demonstrates a simple gradient-based jailbreak attack on a chat language model. It searches for a short adversarial suffix that can be appended to a harmless prompt so the model becomes more likely to generate a chosen target response, then compares the model output and target-token probabilities before and after the optimization.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


# ============================================================
# Configuration
# ============================================================

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

embedding = model.get_input_embeddings()
device = embedding.weight.device

model_vocab_size = embedding.weight.shape[0]
tokenizer_vocab_size = len(tokenizer)

assert tokenizer_vocab_size <= model_vocab_size


# ============================================================
# Test objective
# ============================================================

prompt = "What is the capital of France?"
target = " Penguins live in Antarctica."

suffix_len = 8
top_k = 32
max_new_tokens = 100
num_steps = 20


# ============================================================
# Chat tokenization
# ============================================================

def tokenize_chat_content(content):
    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    return tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids[0].to(device)


# ============================================================
# Determine chat-template tail
# ============================================================

def find_chat_tail(prompt):
    """
    Determine which tokens occur after the user's content.

    We compare:

        prompt

    with:

        prompt + marker

    and find their longest common suffix. That common suffix
    corresponds to the chat-template tokens following the user
    message.
    """

    marker = "XYZ_GCG_MARKER_123456789"

    base_ids = tokenize_chat_content(prompt)
    marked_ids = tokenize_chat_content(prompt + marker)

    n1 = len(base_ids)
    n2 = len(marked_ids)

    common_suffix_len = 0

    while (
        common_suffix_len < n1
        and common_suffix_len < n2
        and base_ids[n1 - 1 - common_suffix_len]
        == marked_ids[n2 - 1 - common_suffix_len]
    ):
        common_suffix_len += 1

    if common_suffix_len == 0:
        raise RuntimeError(
            "Could not locate the chat-template tail."
        )

    chat_tail_ids = base_ids[-common_suffix_len:]
    prompt_prefix_ids = base_ids[:-common_suffix_len]

    return (
        base_ids,
        prompt_prefix_ids,
        chat_tail_ids,
    )


(
    original_chat_ids,
    prompt_prefix_ids,
    chat_tail_ids,
) = find_chat_tail(prompt)


# ============================================================
# Diagnostics
# ============================================================

print("\nChat tail tokens:")
print(
    tokenizer.convert_ids_to_tokens(
        chat_tail_ids.tolist()
    )
)

print("\nPrompt prefix:")
print(
    tokenizer.decode(
        prompt_prefix_ids.tolist()
    )
)

print("\nChat tail:")
print(
    repr(
        tokenizer.decode(
            chat_tail_ids.tolist()
        )
    )
)


# ============================================================
# Target IDs
# ============================================================

target_ids = tokenizer(
    target,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0].to(device)

print("\nTarget IDs:")
print(target_ids.tolist())

print("\nTarget tokens:")
print(
    tokenizer.convert_ids_to_tokens(
        target_ids.tolist()
    )
)


# ============================================================
# Allowed suffix vocabulary
# ============================================================

special_ids = set(tokenizer.all_special_ids)

allowed_token_mask = torch.ones(
    tokenizer_vocab_size,
    dtype=torch.bool,
    device=device,
)

for tok_id in special_ids:
    if 0 <= tok_id < tokenizer_vocab_size:
        allowed_token_mask[tok_id] = False

allowed_token_ids = torch.where(
    allowed_token_mask
)[0]

print(
    f"\nAllowed suffix tokens: "
    f"{len(allowed_token_ids)} / "
    f"{tokenizer_vocab_size}"
)


# ============================================================
# Initialize suffix
# ============================================================

rand_indices = torch.randint(
    low=0,
    high=len(allowed_token_ids),
    size=(suffix_len,),
    device=device,
)

suffix_ids = allowed_token_ids[rand_indices]

# Save initial suffix for before/after comparison.
initial_suffix_ids = suffix_ids.clone()


# ============================================================
# Construct input directly in token space
# ============================================================

def build_input_ids(suffix_ids):
    """
    Insert suffix token IDs directly.

    This avoids:

        suffix IDs
            -> decode()
            -> concatenate text
            -> tokenize()

    which could change the tokenization.
    """

    return torch.cat(
        [
            prompt_prefix_ids,
            suffix_ids,
            chat_tail_ids,
        ],
        dim=0,
    )


suffix_start = len(prompt_prefix_ids)

suffix_positions = torch.arange(
    suffix_start,
    suffix_start + suffix_len,
    device=device,
)


# ============================================================
# Alignment check
# ============================================================

def check_alignment(suffix_ids):
    ids = build_input_ids(suffix_ids)

    actual = ids[suffix_positions]

    if not torch.equal(actual, suffix_ids):
        raise RuntimeError(
            "Suffix alignment failed.\n"
            f"Expected: {suffix_ids.tolist()}\n"
            f"Actual:   {actual.tolist()}"
        )


check_alignment(suffix_ids)


# ============================================================
# Target loss
# ============================================================

@torch.no_grad()
def compute_loss(suffix_ids):
    input_ids = build_input_ids(suffix_ids)

    full_ids = torch.cat(
        [
            input_ids,
            target_ids,
        ],
        dim=0,
    ).unsqueeze(0)

    labels = full_ids.clone()

    # Only compute loss over target tokens.
    labels[:, :input_ids.numel()] = -100

    out = model(
        input_ids=full_ids,
        labels=labels,
    )

    return out.loss


# ============================================================
# Generation
# ============================================================

@torch.no_grad()
def generate(suffix_ids=None):
    if suffix_ids is None:
        input_ids = original_chat_ids
    else:
        input_ids = build_input_ids(suffix_ids)

    attention_mask = torch.ones_like(
        input_ids,
        dtype=torch.long,
    )

    outputs = model.generate(
        input_ids=input_ids.unsqueeze(0),
        attention_mask=attention_mask.unsqueeze(0),
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    new_tokens = outputs[
        0,
        input_ids.numel():
    ]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )


# ============================================================
# Inspect target-token probabilities
# ============================================================

@torch.no_grad()
def inspect_target(suffix_ids):
    input_ids = build_input_ids(suffix_ids)

    full_ids = torch.cat(
        [
            input_ids,
            target_ids,
        ],
        dim=0,
    ).unsqueeze(0)

    out = model(
        input_ids=full_ids,
    )

    logits = out.logits[0]

    # logits[t] predicts token t+1.
    start = input_ids.numel() - 1

    print("\nTarget token probabilities:")

    for j, target_id_tensor in enumerate(target_ids):
        target_id = target_id_tensor.item()

        current_logits = logits[start + j].float()

        probs = torch.softmax(
            current_logits,
            dim=-1,
        )

        prob = probs[target_id].item()

        rank = (
            (
                current_logits
                > current_logits[target_id]
            )
            .sum()
            .item()
            + 1
        )

        top_ids = torch.topk(
            current_logits,
            k=5,
        ).indices

        top_tokens = [
            tokenizer.decode([x.item()])
            for x in top_ids
        ]

        print(
            f"{j:02d} "
            f"target={tokenizer.decode([target_id])!r:18s} "
            f"p={prob:.6f} "
            f"rank={rank:5d} "
            f"top5={top_tokens}"
        )


# ============================================================
# Initial diagnostics
# ============================================================

print("\n========================================")
print("Initial suffix")
print("========================================")

print("IDs:")
print(
    suffix_ids.tolist()
)

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        suffix_ids.tolist()
    )
)

print("\nDecoded:")
print(
    repr(
        tokenizer.decode(
            suffix_ids.tolist()
        )
    )
)

print("\nInitial target loss:")
print(
    compute_loss(
        suffix_ids
    ).item()
)


print("\n========================================")
print("Target probabilities before GCG")
print("========================================")

inspect_target(
    initial_suffix_ids
)


print("\n========================================")
print("Model output before GCG")
print("========================================")

print(
    generate()
)


# ============================================================
# GCG optimization
# ============================================================

for step in range(num_steps):

    check_alignment(suffix_ids)

    input_ids = build_input_ids(
        suffix_ids
    )

    # --------------------------------------------------------
    # Differentiable one-hot suffix
    # --------------------------------------------------------

    one_hot = torch.zeros(
        suffix_len,
        model_vocab_size,
        dtype=embedding.weight.dtype,
        device=device,
    )

    one_hot.scatter_(
        1,
        suffix_ids[:, None],
        1.0,
    )

    one_hot.requires_grad_()

    suffix_embeds = (
        one_hot
        @ embedding.weight
    )

    # --------------------------------------------------------
    # Embed complete prompt
    # --------------------------------------------------------

    base_embeds = (
        embedding(input_ids)
        .detach()
        .clone()
    )

    # Replace exactly the suffix positions with differentiable
    # embeddings.
    base_embeds[
        suffix_positions
    ] = suffix_embeds

    target_embeds = (
        embedding(target_ids)
        .detach()
    )

    inputs_embeds = torch.cat(
        [
            base_embeds,
            target_embeds,
        ],
        dim=0,
    ).unsqueeze(0)

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    full_ids = torch.cat(
        [
            input_ids,
            target_ids,
        ],
        dim=0,
    ).unsqueeze(0)

    labels = full_ids.clone()

    labels[
        :,
        :input_ids.numel()
    ] = -100

    # --------------------------------------------------------
    # Compute gradient
    # --------------------------------------------------------

    model.zero_grad(
        set_to_none=True
    )

    out = model(
        inputs_embeds=inputs_embeds,
        labels=labels,
    )

    loss = out.loss

    loss.backward()

    grad = one_hot.grad

    if grad is None:
        raise RuntimeError(
            "Suffix gradient is None."
        )

    # --------------------------------------------------------
    # Candidate search
    # --------------------------------------------------------

    current_loss = compute_loss(
        suffix_ids
    ).item()

    best_loss = current_loss
    best_pos = None
    best_tok = None

    for pos in range(suffix_len):

        # GCG first-order token score.
        scores = (
            -grad[
                pos,
                :tokenizer_vocab_size
            ]
            .float()
        )

        # Do not choose special/control tokens.
        scores[
            ~allowed_token_mask
        ] = -float("inf")

        candidate_tokens = torch.topk(
            scores,
            k=min(
                top_k,
                len(allowed_token_ids),
            ),
        ).indices

        for tok_tensor in candidate_tokens:

            tok = tok_tensor.item()

            # No reason to evaluate the current token.
            if tok == suffix_ids[pos].item():
                continue

            candidate_suffix = (
                suffix_ids.clone()
            )

            candidate_suffix[pos] = tok

            candidate_loss = (
                compute_loss(
                    candidate_suffix
                )
                .item()
            )

            if candidate_loss < best_loss:

                best_loss = candidate_loss
                best_pos = pos
                best_tok = tok

    # --------------------------------------------------------
    # Apply best token replacement
    # --------------------------------------------------------

    if best_pos is None:

        print(
            f"step={step:03d} "
            f"loss={best_loss:.6f} "
            "(no improving coordinate found)"
        )

        break

    suffix_ids = suffix_ids.clone()

    suffix_ids[
        best_pos
    ] = best_tok

    check_alignment(
        suffix_ids
    )

    decoded_suffix = tokenizer.decode(
        suffix_ids.tolist()
    )

    print(
        f"step={step:03d} "
        f"loss={best_loss:.6f} "
        f"pos={best_pos} "
        f"token={best_tok} "
        f"suffix={decoded_suffix!r}"
    )


# ============================================================
# Final diagnostics
# ============================================================

print("\n========================================")
print("Final suffix")
print("========================================")

print("IDs:")
print(
    suffix_ids.tolist()
)

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        suffix_ids.tolist()
    )
)

print("\nDecoded:")
print(
    repr(
        tokenizer.decode(
            suffix_ids.tolist()
        )
    )
)

print("\nFinal target loss:")
print(
    compute_loss(
        suffix_ids
    ).item()
)


print("\n========================================")
print("Target probabilities AFTER GCG")
print("========================================")

inspect_target(
    suffix_ids
)


print("\n========================================")
print("Model output after GCG")
print("========================================")

print(
    generate(
        suffix_ids
    )
)